# Lab 3: Generate, Check, and Store Data with PhyPhox

This lab introduces the full workflow for working with data collected by students using **PhyPhox**:
1. Generate data in the app.
2. Check the data quality in Python.
3. Store the cleaned dataset in a structured way.

## Learning goals
- Understand how PhyPhox exports measurement files.
- Read exported data with Python.
- Inspect and validate data quality.
- Save processed data so it can be reused later.

## Suggested workflow
- Students collect data using PhyPhox on their phones.
- They export the data as CSV, JSON, or TXT.
- They load the files into this notebook and verify the contents.
- They save a clean version for later analysis.


In [ ]:
# Section 1: Import Required Libraries
import pandas as pd
import numpy as np
import json
from pathlib import Path

print('Libraries imported successfully.')

## Section 2: Set Up File Paths and Experiment Metadata

Use this cell to define where your raw and processed data should be stored. You can change the folder names if needed.

- `raw_data_folder`: contains the exported files from PhyPhox.
- `processed_data_folder`: stores clean CSV/JSON outputs.
- `metadata`: records experiment details such as student name, date, and sensor type.

In [ ]:
# Section 2: Set Up File Paths and Experiment Metadata
from pathlib import Path

project_root = Path.cwd()
raw_data_folder = project_root / 'data' / 'raw_phyphox'
processed_data_folder = project_root / 'data' / 'processed'

raw_data_folder.mkdir(parents=True, exist_ok=True)
processed_data_folder.mkdir(parents=True, exist_ok=True)

metadata = {
    'student_name': 'Student Name',
    'date': 'YYYY-MM-DD',
    'experiment': 'PhyPhox measurement',
    'sensor': 'Light / Acceleration / Gyroscope / etc.'
}

print('Raw folder:', raw_data_folder)
print('Processed folder:', processed_data_folder)
print('Metadata:', metadata)

## Section 3: Read PhyPhox Export Files

In this section, students load the exported data into Python. The notebook uses a small helper to find CSV, JSON, or TXT files in the raw folder.

> Tip: After you export data from PhyPhox, place the file into the `data/raw_phyphox` folder before running the next cell.

In [ ]:
# Section 3: Read PhyPhox Export Files
from pathlib import Path

raw_files = sorted(raw_data_folder.glob('*'))
raw_files = [f for f in raw_files if f.is_file()]

print('Found files:')
for file in raw_files:
    print(' -', file.name)

# Try to read CSV files first
csv_files = [f for f in raw_files if f.suffix.lower() == '.csv']
if csv_files:
    data_path = csv_files[0]
    df = pd.read_csv(data_path)
    print(f'\nLoaded CSV file: {data_path.name}')
    display(df.head())
else:
    print('No CSV file found yet. Please export a CSV from PhyPhox and place it in the raw data folder.')

## Section 4: Inspect the Data Structure

Now that the file is loaded, inspect the table carefully.

Questions to ask:
- How many rows and columns does the dataset have?
- What do the column names mean?
- Are the measurement values in a sensible range?
- Does the time column look correct?

In [ ]:
# Section 4: Inspect the Data Structure
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nData types:')
print(df.dtypes)
print('\nFirst few rows:')
display(df.head())
print('\nSummary statistics:')
display(df.describe(include='all').T)

## Section 5: Check Data Quality

This section focuses on finding problems before analysis.

Check for:
- Missing values
- Duplicate rows
- Strange outliers
- Unusual time jumps
- Units or labels that may be inconsistent

In [ ]:
# Section 5: Check Data Quality
print('Missing values per column:')
print(df.isna().sum())

print('\nDuplicate rows:', df.duplicated().sum())

# Quick numeric checks
numeric_columns = df.select_dtypes(include=[np.number]).columns
for col in numeric_columns:
    series = df[col]
    print(f'\nColumn: {col}')
    print(f'  Min: {series.min()}')
    print(f'  Max: {series.max()}')
    print(f'  Mean: {series.mean()}')

# Optional: show rows with missing values
if df.isna().any().any():
    display(df[df.isna().any(axis=1)].head())

## Section 6: Clean and Standardize the Data

Use this section to make the data easier to analyze.

Possible steps:
- Rename columns to clear names.
- Convert time values to a correct format.
- Remove duplicate rows.
- Keep only columns that you need for the next analysis.

In [ ]:
# Section 6: Clean and Standardize the Data
# Example cleanup steps; adapt these to your dataset.

# Remove exact duplicate rows if needed
df_clean = df.drop_duplicates().copy()

# Rename columns if you want clearer names
# df_clean = df_clean.rename(columns={
#     'Time (s)': 'time_s',
#     'Illuminance (lx)': 'illuminance_lx'
# })

# Convert a time column if present and if it looks numeric
if 'Time (s)' in df_clean.columns:
    df_clean['Time (s)'] = pd.to_numeric(df_clean['Time (s)'], errors='coerce')

# Keep a small set of relevant columns if needed
# df_clean = df_clean[['time_s', 'illuminance_lx']]

print('Cleaned dataset shape:', df_clean.shape)
display(df_clean.head())

## Section 7: Store the Processed Data

Save the cleaned dataset in a consistent way so others can reuse it.

Recommended outputs:
- CSV file for easy spreadsheet use.
- JSON file if you want a structured text format.
- A metadata file that records experiment details.

In [ ]:
# Section 7: Store the Processed Data
from datetime import datetime

output_name = f"{metadata['student_name'].replace(' ', '_')}_{metadata['date']}_processed"
csv_output = processed_data_folder / f'{output_name}.csv'
json_output = processed_data_folder / f'{output_name}.json'
metadata_output = processed_data_folder / f'{output_name}_metadata.json'

# Save cleaned data
if 'df_clean' in locals():
    df_clean.to_csv(csv_output, index=False)
    df_clean.to_json(json_output, orient='records', indent=2)
    with metadata_output.open('w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2)

    print('Saved CSV:', csv_output)
    print('Saved JSON:', json_output)
    print('Saved metadata:', metadata_output)
else:
    print('No cleaned dataset found yet. Run the cleaning section first.')

## Section 8: Verify Saved Outputs

This final step checks that the saved files really contain the intended data.

Run the next cell to read the saved CSV and JSON files back and compare them with the cleaned table.

In [ ]:
# Section 8: Verify Saved Outputs
if csv_output.exists():
    df_saved_csv = pd.read_csv(csv_output)
    print('CSV loaded successfully.')
    display(df_saved_csv.head())
    print('CSV shape:', df_saved_csv.shape)

if json_output.exists():
    with json_output.open('r', encoding='utf-8') as f:
        df_saved_json = json.load(f)
    print('\nJSON loaded successfully.')
    print('JSON entries:', len(df_saved_json))
    print('First JSON entry:')
    print(df_saved_json[0] if df_saved_json else 'No entries found.')